In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model="gpt-5.6-terra",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

loader = TextLoader("./files/document.txt")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

docs = loader.load_and_split(text_splitter=splitter)
len(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

cache_dir = LocalFileStore("./.cache/embeddings/")

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    cache_dir,
    namespace=embeddings.model,
)

vectorstore = FAISS.from_documents(docs, cache_embeddings)
retriever = vectorstore.as_retriever()

memory = ConversationBufferMemory(return_messages=True)

def load_memory(_):
    return memory.load_memory_variables({})["history"]

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 도움이 되는 어시스턴트 입니다.
    아래 문서 내용(context)만 사용해서 질문에 답하세요.
    답을 모른다면 모른다고 답하고, 지어내지 마세요. 답변은 한국어로 하세요.

    {context}"""),
        MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
        "history": RunnableLambda(load_memory),
    }
    | prompt
    | chat
)

def invoke_chain(question):
    result = chain.invoke(question)
    memory.save_context({"input": question}, {"output": result.content})

/var/folders/9d/nwpdspds2ws84zl4lpbyhzg00000gn/T/ipykernel_19743/6058756.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/namseonglyong/Project/fullstack-gpt/env/lib/python3.15/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()
/var/folders/9d/n

본문에서 윈스턴은 처음에는 아론슨이 무죄임을 보여주는 사진을 보았다고 기억합니다. 그러나 당의 세뇌 이후에는 그 사진은 존재하지 않았고 자신이 지어낸 기억이며, 아론슨은 기소된 범죄에 유죄였다고 받아들입니다.  

따라서 본문상 당의 주장에 따르면 유죄이지만, 실제로 유죄인지 확정할 수는 없습니다.그는 테이블 위에 **“2 + 2 = 5”**라고 썼습니다.줄리아는 윈스턴이 사랑했던 여성입니다. 둘은 당에 맞서는 비밀스러운 관계를 맺었지만, 결국 줄리아도 체포되어 고문을 받았고, 윈스턴은 101호실에서 쥐의 공포 때문에 “줄리아에게 해라!”라고 외치며 그녀를 배신합니다.

In [2]:
invoke_chain("Aaronson 은 유죄인가요?")

본문에서는 당이 아론슨이 유죄라고 주장하며, 윈스턴도 세뇌 뒤에는 그렇게 받아들입니다. 그러나 윈스턴이 과거에 그의 무죄를 입증하는 사진을 보았다고 기억한 점을 고려하면, 실제로 유죄였는지는 본문만으로 확정할 수 없습니다.

In [3]:
invoke_chain("그가 테이블에 어떤 메시지를 썼나요?")

제공된 본문에는 그가 테이블에 쓴 메시지가 나오지 않습니다. 따라서 알 수 없습니다.

In [4]:
invoke_chain("Julia 는 누구인가요?")

줄리아는 윈스턴이 사랑했던 여성입니다. 윈스턴은 그녀를 깊이 사랑했지만, 101호실에서 쥐에게 공포를 느끼자 자신의 벌을 그녀에게 돌리려 하며 “줄리아에게 해라! 나 말고 줄리아에게!”라고 외칩니다. 이후 둘은 다시 만나지만, 서로가 상대를 배신했다는 사실 때문에 예전과 같은 감정을 느끼지 못합니다.